# 05 – E3 / E4: CLIP + prompt + LayerNorm (Thành viên B)
**GPU T4, Internet On.** Add Data: Sketchy gốc, Flickr15K gốc, `sbir-cache-256-photos`.
- **E4** = cấu hình bài báo: `LAMBDA = 0.5`. **E3** = ablation bỏ L_cls: `LAMBDA = 0`. Chạy E3 và E4 trên **2 tài khoản Kaggle song song** bằng cùng notebook này.
- Mỗi run: 60 epoch trên 94 lớp train, chấm mAP@200 trên 10 lớp val sau mỗi epoch, giữ checkpoint tốt nhất, cuối cùng đánh giá **một lần** trên 21 lớp test (+ Flickr15K). Ước tính 5–7 giờ → dùng **Save Version → Save & Run All**.
- Bị ngắt phiên: thêm output của lần chạy trước làm input, điền `RESUME_FROM` = đường dẫn `prompts_last.pt`, chạy lại.
- Sketch đọc từ dữ liệu gốc, ảnh thật đọc từ cache (mục 2.4 kế hoạch).

In [ ]:
# ======= CẤU HÌNH =======
EXP     = "E4"     # "E4" (đầy đủ, cấu hình bài báo) | "E3" (ablation bỏ L_cls)
LAMBDA  = 0.5      # E4: 0.5 · E3: 0
EPOCHS  = 60       # theo bài báo
EXTRA   = ""       # tham số thêm cho train.py nếu cần (thường để trống)
RESUME_FROM = ""   # ví dụ "/kaggle/input/<output-lan-truoc>/checkpoints/E4/prompts_last.pt"
SKETCHY_RAW = "/kaggle/input/datasets/hongonthi/sketchyextended/sketchy"
FLICKR_RAW  = "/kaggle/input/datasets/hongonthi/flickr15k/Flickr15K"
CACHE_ROOT  = "/kaggle/input/datasets/hongonthi/sbir-cache-256-photos"   # "" = đọc ảnh gốc
REPO        = "https://github.com/thaihoang104569/CS406.git"
# =========================
import os
for n, p in [("SKETCHY_RAW", SKETCHY_RAW), ("FLICKR_RAW", FLICKR_RAW)] + ([("CACHE_ROOT", CACHE_ROOT)] if CACHE_ROOT else []):
    assert p and os.path.isdir(p), f"{n} không tồn tại: {p!r}"
assert not RESUME_FROM or os.path.isfile(RESUME_FROM), f"RESUME_FROM không tồn tại: {RESUME_FROM}"
WORK = "/kaggle/working/CS406"
if os.path.isdir(WORK):
    !git -C {WORK} pull -q
else:
    !git clone -q {REPO} {WORK}
!git -C {WORK} log --oneline -1
SK_PHOTO = f"{CACHE_ROOT}/sketchy" if CACHE_ROOT else SKETCHY_RAW
FL_PHOTO = f"{CACHE_ROOT}/flickr15k" if CACHE_ROOT else FLICKR_RAW
OUT = "/kaggle/working"

In [ ]:
!pip install -q open_clip_torch
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Split cố định (đọc dữ liệu gốc, đúng 10 lớp val đã commit)

In [ ]:
!python {WORK}/scripts/prepare_data.py --sketchy_root "{SKETCHY_RAW}" --flickr_root "{FLICKR_RAW}" \
    --out_dir {OUT}/data_prep --class_split_csv {WORK}/data/splits/sketchy_class_splits.csv \
    --skip_dup_check --no_figures --sample_per_group 20
SPL = f"{OUT}/data_prep/splits"
COMMON = (f'--sketchy_csv {SPL}/sketchy_zero_shot_splits.csv --sketchy_root "{SKETCHY_RAW}" --sketchy_photo_root "{SK_PHOTO}" '
          f'--flickr_csv {SPL}/flickr15k_splits.csv --flickr_root "{FLICKR_RAW}" --flickr_photo_root "{FL_PHOTO}" '
          f'--ckpt_dir {OUT}/checkpoints --results_dir {OUT}/results --feat_dir {OUT}/features --workers 4')

## 2. Chạy thử `--debug` (~3 phút: 20 lớp, 2 epoch, có đánh giá test thu nhỏ)

In [ ]:
!cd {WORK} && python -m sbir.deep.train --exp {EXP} --lambda_cls {LAMBDA} {EXTRA} {COMMON} --debug --eval_test

## 3. Chạy đầy đủ (60 epoch + đánh giá test một lần)

In [ ]:
RES = f"--resume {RESUME_FROM}" if RESUME_FROM else ""
!cd {WORK} && python -m sbir.deep.train --exp {EXP} --lambda_cls {LAMBDA} --epochs {EPOCHS} {EXTRA} {COMMON} {RES} --eval_test

## 4. Kết quả

In [ ]:
import json, shutil, pandas as pd, matplotlib.pyplot as plt
m = json.load(open(f"{OUT}/results/{EXP}/metrics.json"))
print("best epoch:", m["best_epoch"], "| best val mAP@200:", m["best_val_mAP@200"], "| trainable:", m["trainable_params"])
h = pd.DataFrame([{**{k: r[k] for k in ("epoch", "loss", "l_tri", "l_cls")}, "val_mAP@200": r.get("val", {}).get("mAP@200")} for r in m["history"]])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
h.plot(x="epoch", y=["loss", "l_tri", "l_cls"], ax=ax[0], title="training loss")
h.plot(x="epoch", y="val_mAP@200", ax=ax[1], title="val mAP@200 (10 lớp)")
plt.tight_layout(); plt.show()
display(pd.DataFrame({k: v["visual"] for k, v in m["results"].items()}))
display(pd.DataFrame(m["results"]["sketchy_test"]["per_class_visual"]).T.sort_values("mAP@200"))
for d in ("results", "checkpoints", "features"):
    shutil.make_archive(f"{OUT}/{EXP}_{d}", "zip", OUT, f"{d}/{EXP}")
!ls -la {OUT}/*.zip